# Pre-entrega 3: Sistema de recuperación semántica local (RAG)
Ingesta de documentos en ChromaDB, recuperación por similitud y generación de respuestas basadas solo en el contexto recuperado.

## 1. Instalación de librerías

In [1]:
!pip install -q langchain-core langchain-text-splitters langchain-chroma langchain-huggingface langchain-google-genai chromadb sentence-transformers tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95

## 2. Configuración

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

GEMINI_MODEL = "gemini-3.8-flash"
EMBEDDINGS_MODEL = "intfloat/multilingual-e5-small"   # el mismo para indexar y consultar

DATA_DIR = "data"
VECTORSTORE_DIR = "vectorstore"
COLLECTION_NAME = "apuntes_curso"

CHUNK_SIZE = 500      # tokens
CHUNK_OVERLAP = 50    # tokens
TOP_K = 4             # fragmentos a recuperar (entre 3 y 5)

os.makedirs(DATA_DIR, exist_ok=True)

## 3. Configuración de logs

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
for ruidoso in ("httpx", "chromadb", "sentence_transformers", "google_genai"):
    logging.getLogger(ruidoso).setLevel(logging.WARNING)
logger = logging.getLogger("rag")

## 4. Dataset de ejemplo
Tres archivos de apuntes del curso que forman la base de conocimiento.

In [4]:
%%writefile data/pydantic_validacion.txt
Pydantic: validación de datos en Python

Pydantic es una librería de Python que permite definir la forma de los datos mediante clases que heredan de BaseModel. Cada atributo de la clase lleva una anotación de tipo, y Pydantic verifica en tiempo de ejecución que los valores recibidos cumplan con ese tipo. Si un valor no coincide, se lanza una excepción ValidationError que detalla qué campo falló y por qué.

Restricciones con Field
La función Field permite agregar restricciones a cada campo. Por ejemplo, min_length y max_length limitan el largo de un texto o de una lista, mientras que ge y le establecen valores mínimos y máximos para números. En el curso usamos Field para que la temperatura de un modelo esté entre 0 y 2, y para que la lista de tecnologías extraídas tenga al menos un elemento.

Validadores personalizados
Cuando las restricciones de Field no alcanzan, se puede escribir un validador con el decorador field_validator. El validador recibe el valor, puede transformarlo (por ejemplo, quitar espacios o duplicados) y debe devolverlo. Si el valor es inválido, el validador lanza un ValueError y Pydantic lo convierte en un ValidationError.

Enums
Para campos que solo aceptan un conjunto cerrado de valores conviene usar una clase Enum. En la Pre-entrega 2 el nivel de criticidad se definió como un Enum con tres valores posibles: baja, media y alta. Cualquier otro valor, como "urgente", es rechazado antes de llegar al resto del programa.

Serialización
Los modelos de Pydantic se convierten a diccionario con model_dump y a JSON con model_dump_json. El parámetro mode="json" hace que los Enum se conviertan a texto, lo que facilita imprimir o guardar los resultados.

Por qué validar
Los modelos de lenguaje no siempre devuelven lo que se les pide. Validar la salida con Pydantic permite detectar respuestas incompletas o mal formadas y decidir si reintentar la llamada. Es importante recordar que la validación garantiza la estructura de los datos, pero no la calidad del contenido: un modelo puede devolver un JSON perfectamente válido con información incorrecta.

Writing data/pydantic_validacion.txt


In [5]:
%%writefile data/langchain_lcel.txt
LangChain y LCEL

LangChain es un framework para construir aplicaciones con modelos de lenguaje. Su pieza central es LCEL (LangChain Expression Language), una forma de componer pasos usando el operador | (pipe). Cada paso es un Runnable: recibe una entrada, la procesa y pasa el resultado al siguiente.

Prompt templates
ChatPromptTemplate define los mensajes que se envían al modelo, con variables entre llaves como {pregunta} o {contexto}. LangChain completa esas variables al momento de ejecutar la cadena. Por eso se recomienda no usar f-strings de Python para armar prompts: el template mantiene el prompt modular y deja que LangChain gestione las variables de entrada. El método partial permite fijar de antemano algunas variables, como las instrucciones de formato.

Modelos y salida estructurada
Los modelos de chat, como ChatGoogleGenerativeAI o ChatOpenAI, también son Runnables. Su salida puede convertirse en un objeto usando un output parser. PydanticOutputParser genera instrucciones de formato a partir de un modelo Pydantic y luego valida el texto devuelto. Otra alternativa es with_structured_output, que usa las capacidades nativas del proveedor para devolver datos estructurados.

Ejecución asíncrona
Todo Runnable tiene un método invoke para ejecución sincrónica y un método ainvoke para ejecución asíncrona. Con ainvoke y asyncio.gather se pueden procesar varias consultas al mismo tiempo sin que el programa quede bloqueado esperando cada respuesta.

Reintentos
El método with_retry envuelve un Runnable y lo vuelve a ejecutar si lanza ciertas excepciones. Se configura con retry_if_exception_type, para indicar qué errores se reintentan, y stop_after_attempt, para fijar la cantidad máxima de intentos. En la Pre-entrega 2 se usaron 3 intentos en total con espera exponencial entre uno y otro.

finish_reason
Cada respuesta del modelo incluye metadatos con el motivo de finalización. Si el valor indica un corte por límite de tokens, como MAX_TOKENS en Gemini o length en OpenAI, la respuesta puede estar incompleta y conviene reintentar en lugar de procesarla.

Writing data/langchain_lcel.txt


In [6]:
%%writefile data/rag_chromadb.txt
RAG y bases vectoriales con ChromaDB

RAG (Retrieval-Augmented Generation) es una técnica que combina búsqueda de información con generación de texto. En lugar de depender solo de lo que el modelo aprendió durante su entrenamiento, el sistema busca fragmentos relevantes en una base de documentos propia y se los pasa al modelo como contexto para responder.

Embeddings
Un embedding es un vector de números que representa el significado de un texto. Textos con significado parecido producen vectores cercanos. El error más común en RAG es usar un modelo de embeddings para indexar los documentos y otro distinto para las consultas: los vectores quedan en espacios diferentes y la búsqueda devuelve resultados sin sentido.

Chunking
Antes de indexar, los documentos se dividen en fragmentos llamados chunks. RecursiveCharacterTextSplitter intenta cortar primero por párrafos, después por líneas y finalmente por palabras, para no partir ideas a la mitad. En el curso se recomienda un tamaño de 500 tokens con un solapamiento (overlap) de 50 tokens, de modo que la información que queda en el borde de un chunk también aparezca al principio del siguiente.

ChromaDB
ChromaDB es una base de datos vectorial que puede funcionar de forma local. Con un cliente persistente, los vectores se guardan en una carpeta del disco, por ejemplo ./vectorstore, y no hace falta volver a indexar cada vez que se ejecuta el programa. Un buen script de ingesta verifica si la colección ya tiene datos antes de procesar los documentos otra vez.

Recuperación
Cuando llega una pregunta, se calcula su embedding y se buscan los chunks más cercanos. La cantidad de fragmentos recuperados se llama top_k. Conviene mantenerlo entre 3 y 5: pasar demasiados fragmentos al modelo puede superar el límite de tokens y degradar la atención, un problema conocido como Lost in the Middle.

Generación grounded
El prompt del sistema debe indicar al modelo que responda solo con la información del contexto y que diga "No lo sé" si la respuesta no está ahí. Esto reduce las alucinaciones. Además, pedir que el modelo indique las fuentes que usó permite verificar de dónde salió cada respuesta.

Writing data/rag_chromadb.txt


In [7]:
for archivo in sorted(os.listdir(DATA_DIR)):
    ruta = os.path.join(DATA_DIR, archivo)
    print(f"{archivo}: {len(open(ruta, encoding='utf-8').read())} caracteres")

langchain_lcel.txt: 2085 caracteres
pydantic_validacion.txt: 2089 caracteres
rag_chromadb.txt: 2158 caracteres


## 5. Modelo de embeddings
Se usa el mismo modelo para indexar los documentos y para convertir las preguntas.

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDINGS_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

vector = embeddings.embed_query("¿Qué es un embedding?")
logger.info("Embeddings listos: %s (dimensión %d)", EMBEDDINGS_MODEL, len(vector))

23:02:24 | INFO    | numexpr.utils | NumExpr defaulting to 2 threads.
23:02:35 | INFO    | datasets | TensorFlow version 2.21.0 available.
23:02:35 | INFO    | datasets | JAX version 0.11.1 available.
23:02:41 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

23:02:52 | INFO    | rag | Embeddings listos: intfloat/multilingual-e5-small (dimensión 384)


## 6. Carga de documentos desde /data

In [9]:
from langchain_core.documents import Document


def cargar_documentos(carpeta: str = DATA_DIR) -> list[Document]:
    """Lee los archivos .txt y .md de la carpeta y los convierte en Documents."""
    documentos = []
    for archivo in sorted(os.listdir(carpeta)):
        if archivo.endswith((".txt", ".md")):
            with open(os.path.join(carpeta, archivo), encoding="utf-8") as f:
                documentos.append(Document(page_content=f.read(), metadata={"fuente": archivo}))
    logger.info("Documentos cargados: %d", len(documentos))
    return documentos


documentos = cargar_documentos()

23:04:25 | INFO    | rag | Documentos cargados: 3


## 7. Chunking (500 tokens con 50 de overlap)

In [10]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
encoder = tiktoken.get_encoding("cl100k_base")


def fragmentar(documentos: list[Document]) -> list[Document]:
    chunks = splitter.split_documents(documentos)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    logger.info("Chunks generados: %d", len(chunks))
    return chunks


chunks = fragmentar(documentos)
for c in chunks:
    print(f"{c.metadata['fuente']:<25} chunk {c.metadata['chunk_id']}: "
          f"{len(encoder.encode(c.page_content))} tokens")

23:04:53 | INFO    | rag | Chunks generados: 5


langchain_lcel.txt        chunk 0: 495 tokens
pydantic_validacion.txt   chunk 1: 414 tokens
pydantic_validacion.txt   chunk 2: 92 tokens
rag_chromadb.txt          chunk 3: 430 tokens
rag_chromadb.txt          chunk 4: 73 tokens


## 8. Ingesta en ChromaDB (con persistencia)
Si la colección ya tiene datos, no se vuelve a indexar.

In [11]:
from langchain_chroma import Chroma


def ingestar(forzar: bool = False) -> Chroma:
    """Crea o abre la colección persistente y la puebla solo si está vacía."""
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=VECTORSTORE_DIR,
        collection_metadata={"hnsw:space": "cosine"},
    )

    existentes = vectorstore._collection.count()
    if existentes > 0 and not forzar:
        logger.info("La colección '%s' ya existe con %d chunks -> se omite la ingesta",
                    COLLECTION_NAME, existentes)
        return vectorstore

    if forzar and existentes > 0:
        logger.info("Reindexando: se borra la colección anterior")
        vectorstore.reset_collection()

    chunks = fragmentar(cargar_documentos())
    ids = [f"{c.metadata['fuente']}-{c.metadata['chunk_id']}" for c in chunks]
    vectorstore.add_documents(chunks, ids=ids)
    logger.info("Ingesta completa: %d chunks guardados en ./%s", len(chunks), VECTORSTORE_DIR)
    return vectorstore


vectorstore = ingestar()

23:05:17 | INFO    | rag | Documentos cargados: 3
23:05:17 | INFO    | rag | Chunks generados: 5
23:05:20 | INFO    | rag | Ingesta completa: 5 chunks guardados en ./vectorstore


## 9. Prueba de persistencia
Al llamar de nuevo a la ingesta, detecta que la base ya existe y no reindexa.

In [12]:
vectorstore = ingestar()

23:05:40 | INFO    | rag | La colección 'apuntes_curso' ya existe con 5 chunks -> se omite la ingesta


## 10. Prueba de búsqueda por similitud
Menor distancia = más parecido a la pregunta.

In [13]:
pregunta = "¿Cuántos fragmentos conviene recuperar y por qué?"
resultados = vectorstore.similarity_search_with_score(pregunta, k=TOP_K)

for doc, distancia in resultados:
    print(f"[{distancia:.3f}] {doc.metadata['fuente']} (chunk {doc.metadata['chunk_id']})")
    print("   ", doc.page_content[:120].replace("\n", " "), "...\n")

[0.154] rag_chromadb.txt (chunk 3)
    RAG y bases vectoriales con ChromaDB  RAG (Retrieval-Augmented Generation) es una técnica que combina búsqueda de inform ...

[0.192] pydantic_validacion.txt (chunk 2)
    Por qué validar Los modelos de lenguaje no siempre devuelven lo que se les pide. Validar la salida con Pydantic permite  ...

[0.202] pydantic_validacion.txt (chunk 1)
    Pydantic: validación de datos en Python  Pydantic es una librería de Python que permite definir la forma de los datos me ...

[0.206] langchain_lcel.txt (chunk 0)
    LangChain y LCEL  LangChain es un framework para construir aplicaciones con modelos de lenguaje. Su pieza central es LCE ...



## 11. Esquema de la respuesta (Pydantic)

In [14]:
from pydantic import BaseModel, Field, model_validator


class RespuestaRAG(BaseModel):
    """Respuesta del asistente basada solo en el contexto recuperado."""

    respuesta: str = Field(..., min_length=2,
                           description='Respuesta en español. Si no está en el contexto: "No lo sé".')
    encontrado: bool = Field(..., description="True si la respuesta está en el contexto, False si no.")
    referencias: list[str] = Field(default_factory=list,
                                   description="Nombres de los archivos fuente usados para responder.")

    @model_validator(mode="after")
    def sin_referencias_si_no_sabe(self):
        if not self.encontrado:
            self.referencias = []
        return self

## 12. Modelo de lenguaje (Gemini)

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0, max_retries=5)
logger.info("LLM configurado: %s", GEMINI_MODEL)

23:08:38 | INFO    | rag | LLM configurado: gemini-3.8-flash


## 13. Prompt de sistema ("filtro de veracidad") y output parser

In [16]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = PydanticOutputParser(pydantic_object=RespuestaRAG)

PROMPT_RAG = ChatPromptTemplate.from_messages([
    ("system",
     "Sos un asistente técnico. Respondé SOLO con la información del CONTEXTO.\n"
     "Reglas:\n"
     "- Si la respuesta no está en el contexto, respondé exactamente \"No lo se\" "
     "y poné encontrado en false.\n"
     "- No uses conocimiento propio ni inventes datos.\n"
     "- En referencias, poné solo los nombres de archivo (Fuente) que usaste.\n\n"
     "{format_instructions}"),
    ("human", "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"),
]).partial(format_instructions=parser.get_format_instructions())


def formatear_contexto(docs: list[Document]) -> str:
    """Une los fragmentos recuperados indicando de qué archivo viene cada uno."""
    return "\n\n---\n\n".join(
        f"[Fuente: {d.metadata['fuente']}]\n{d.page_content}" for d in docs
    )

## 14. Cadena RAG con LCEL
Retriever -> contexto -> prompt -> LLM -> PydanticOutputParser, con reintento si el JSON sale mal formado.

In [17]:
from langchain_core.exceptions import OutputParserException
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

generacion = (
    RunnableLambda(lambda x: {"contexto": formatear_contexto(x["docs"]), "pregunta": x["pregunta"]})
    | PROMPT_RAG
    | llm
    | parser
).with_retry(
    retry_if_exception_type=(OutputParserException,),
    wait_exponential_jitter=True,
    stop_after_attempt=3,
)

cadena_rag = (
    RunnableParallel(docs=retriever, pregunta=RunnablePassthrough())
    | RunnablePassthrough.assign(resultado=generacion)
)

## 15. Función asíncrona get_rag_response()

In [18]:
async def get_rag_response(query: str) -> RespuestaRAG:
    """Busca en ChromaDB, arma el prompt con el contexto y devuelve una respuesta validada."""
    if not query or not query.strip():
        raise ValueError("La consulta está vacía")

    logger.info("Consulta: %s", query)
    try:
        salida = await cadena_rag.ainvoke(query)
    except Exception as error:
        logger.error("Falló la cadena RAG: %s: %s", type(error).__name__, error)
        raise

    fuentes = {d.metadata["fuente"] for d in salida["docs"]}
    logger.info("Fragmentos recuperados: %d de %s", len(salida["docs"]), sorted(fuentes))

    respuesta: RespuestaRAG = salida["resultado"]
    inventadas = [r for r in respuesta.referencias if r not in fuentes]
    if inventadas:
        logger.warning("Referencias que no estaban en el contexto (se descartan): %s", inventadas)
        respuesta.referencias = [r for r in respuesta.referencias if r in fuentes]

    logger.info("Respuesta validada: encontrado=%s, referencias=%s",
                respuesta.encontrado, respuesta.referencias)
    return respuesta

## 16. Prueba 1: pregunta con respuesta en los documentos

In [19]:
import json

r1 = await get_rag_response("¿Qué hace with_retry y cuántos intentos se usaron en la Pre-entrega 2?")
print(json.dumps(r1.model_dump(), ensure_ascii=False, indent=2))

23:09:57 | INFO    | rag | Consulta: ¿Qué hace with_retry y cuántos intentos se usaron en la Pre-entrega 2?
23:10:51 | INFO    | rag | Fragmentos recuperados: 4 de ['langchain_lcel.txt', 'pydantic_validacion.txt', 'rag_chromadb.txt']
23:10:51 | INFO    | rag | Respuesta validada: encontrado=True, referencias=['langchain_lcel.txt']


{
  "respuesta": "El método with_retry envuelve un Runnable y lo vuelve a ejecutar si lanza ciertas excepciones. En la Pre-entrega 2 se usaron 3 intentos en total con espera exponencial entre uno y otro.",
  "encontrado": true,
  "referencias": [
    "langchain_lcel.txt"
  ]
}


## 17. Prueba 2: pregunta trampa (no está en los documentos)

In [21]:
import asyncio

async def probar(query: str, intentos: int = 3, espera: int = 30):
    for intento in range(1, intentos + 1):
        try:
            r = await get_rag_response(query)
            print(json.dumps(r.model_dump(), ensure_ascii=False, indent=2))
            return r
        except Exception as e:
            if "503" in str(e) and intento < intentos:
                logger.warning("Gemini saturado (503). Reintento %d de %d en %d segundos...",
                               intento + 1, intentos, espera)
                await asyncio.sleep(espera)
            else:
                print(f" Error controlado ({type(e).__name__}): {e}")
                return None


r2 = await probar("¿Cuánto cuesta la API de OpenAI por millón de tokens?")

23:12:16 | INFO    | rag | Consulta: ¿Cuánto cuesta la API de OpenAI por millón de tokens?
23:13:18 | INFO    | rag | Fragmentos recuperados: 4 de ['langchain_lcel.txt', 'pydantic_validacion.txt', 'rag_chromadb.txt']
23:13:18 | INFO    | rag | Respuesta validada: encontrado=False, referencias=[]


{
  "respuesta": "No lo sé",
  "encontrado": false,
  "referencias": []
}
